In [1]:
from datasets import load_dataset

/home/teoaivalis/.conda/envs/artbench_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Load Artbench Dataset

In [2]:
import pandas as pd

df = pd.read_csv('/home/teoaivalis/.cache/kagglehub/datasets/alexanderliao/artbench10/versions/2/ArtBench-10.csv')
#df = pd.read_csv('../../../../codes/artbench/ArtBench-10.csv')
df.head()

,name,artist,url,is_public_domain,length,width,label,split,cifar_index
0,frank-omeara_towards-night-and-winter.jpg,frank-omeara,https://uploads5.wikiart.org/00316/images/fran...,True,800,657,impressionism,train,43186
1,goldstein-grigoriy_morning.jpg,goldstein-grigoriy,https://uploads5.wikiart.org/images/grigoriy-g...,True,521,499,impressionism,train,41151
2,georges-lemmen_man-reading.jpg,georges-lemmen,https://uploads6.wikiart.org/images/georges-le...,True,800,612,impressionism,train,9754
3,theodor-aman_port-of-constantza-1882.jpg,theodor-aman,https://uploads6.wikiart.org/images/theodor-am...,True,560,336,impressionism,train,44244
4,niccolo-cannicci_il-passo-della-futa-1914.jpg,niccolo-cannicci,https://uploads3.wikiart.org/images/niccolo-ca...,True,2400,2322,impressionism,train,46885


In [3]:
print(df.iloc[0][0])
print(df.iloc[0][1])
print(df.iloc[0][2])

print(df.iloc[1][0])
print(df.iloc[1][1])
print(df.iloc[1][2])

print(df.iloc[2][0])
print(df.iloc[2][1])
print(df.iloc[2][2])

print(df.iloc[3][0])
print(df.iloc[3][1])
print(df.iloc[3][2])

frank-omeara_towards-night-and-winter.jpg
frank-omeara
https://uploads5.wikiart.org/00316/images/frank-omeara/towards-night-and-winter.jpg
goldstein-grigoriy_morning.jpg
goldstein-grigoriy
https://uploads5.wikiart.org/images/grigoriy-goldstein/morning.jpg
georges-lemmen_man-reading.jpg
georges-lemmen
https://uploads6.wikiart.org/images/georges-lemmen/man-reading.jpg
theodor-aman_port-of-constantza-1882.jpg
theodor-aman
https://uploads6.wikiart.org/images/theodor-aman/port-of-constantza-1882.jpg


/tmp/ipykernel_3710156/2401613183.py:1: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  print(df.iloc[0][0])
/tmp/ipykernel_3710156/2401613183.py:2: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  print(df.iloc[0][1])
/tmp/ipykernel_3710156/2401613183.py:3: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  print(df.iloc[0][2])
/tmp/ipykernel_3710156/2401613183.py:5: FutureWarning: Series.__getitem__ treating keys as positions is deprecated.

In [4]:
import requests
from bs4 import BeautifulSoup
import time
import pandas as pd
from tqdm.notebook import tqdm 

def scrape_wikiart_artist(artist_slug):
    # Ensure artist_slug is a string
    if not isinstance(artist_slug, str):
        return {"artist_slug": artist_slug, "status": "Invalid Name"}
        
    url = f"https://www.wikiart.org/en/{artist_slug}"
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'}
    
    try:
        response = requests.get(url, headers=headers, timeout=10)
        if response.status_code != 200:
            return {"artist_slug": artist_slug, "status": f"HTTP {response.status_code}"}
        
        soup = BeautifulSoup(response.text, 'html.parser')
        metadata = {"artist_slug": artist_slug, "status": "Success"}
        
        # WikiArt metadata is usually in 'p-info-list' or 'li' tags
        info_list = soup.find('ul', class_='p-info-list')
        items = info_list.find_all('li') if info_list else soup.find_all('li')
        
        for li in items:
            text = li.get_text(separator=" ").strip()
            if ":" in text and len(text) < 200:
                parts = text.split(":", 1)
                key = parts[0].strip().lower()
                val = parts[1].strip()
                metadata[key] = val
        
        return metadata
    except Exception as e:
        return {"artist_slug": artist_slug, "status": f"Error: {str(e)}"}

In [5]:
from tqdm import tqdm  # Change this from tqdm.notebook to just tqdm
# 1. Get unique artist names from your dataframe
unique_artists = df['artist'].unique()
print(f"Total unique artists found: {len(unique_artists)}")

# 2. Test with a small batch first (remove .head(20) for the full run)
test_batch = unique_artists[1000:] 

results = []
for artist in tqdm(test_batch, desc="Scraping WikiArt"):
#for artist in tqdm(unique_artists, desc="Scraping WikiArt"):
    data = scrape_wikiart_artist(artist)
    results.append(data)
    time.sleep(1.1) # polite delay

# 3. Create metadata dataframe
wiki_metadata_df2 = pd.DataFrame(results)

# 4. Display the results
print("\n--- Scraped Metadata ---")
display(wiki_metadata_df1.head())

Total unique artists found: 2126


Scraping WikiArt: 100%|██████████| 1126/1126 [42:59<00:00,  2.29s/it]


--- Scraped Metadata ---


NameError: name 'wiki_metadata_df1' is not defined

In [6]:
# 4. Display the results
print("\n--- Scraped Metadata ---")
display(wiki_metadata_df2.head())


--- Scraped Metadata ---


,artist_slug,status,born,died,nationality,art movement,painting school,field,teachers,art institution,...,san marco,piazza san marco,grand canal,untitled (from conspiracy,three beauties,shinagawa,monday,november,sino-japanese war,sixty-nine stations of the kisokaido
0,vasyl-yermylov,Success,"March 22, 1894 ; Kharkiv, Ukraine","January 6, 1968 ; Kharkiv, Ukraine",Ukrainians,"Avant-garde , \n Constructivism , \n Cubo-Futu...",ARMU (Association of Revolutionary Masters of ...,"painting , \n printmaking , \n design","Ilya Mashkov , \n Pyotr Konchalovsky","Moscow School of Painting, Sculpture and Archi...",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,mario-eloy,Success,"March 15, 1900 ; Algés, Portugal","September 5, 1951 ; Lisbon, Portugal",Portuguese,Expressionism,NaN,painting,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,botong-francisco,Success,"November 4, 1912 ; Angono, Philippines","March 31, 1969 ; Angono, Philippines",Filipino,"Social Realism , \n Modernism",NaN,painting,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,max-bill,Success,"December 22, 1908 ; Winterthur, Switzerland","December 9, 1994 ; Berlin, Germany",Swiss,Concrete Art (Concretism),"Bauhaus , \n Abstraction-Création","painting , \n design , \n architecture , \n dr...",NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,jeanne-hebuterne,Success,"April 6, 1898 ; Paris, France","January 25, 1920 ; Paris, France",French,Expressionism,NaN,painting,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [10]:
frames = [wiki_metadata_df, wiki_metadata_df1]

result = pd.concat(frames)

In [11]:
display(result.head())

,artist_slug,status,born,died,nationality,art movement,field,influenced by,teachers,friends and co-workers,...,ivan bakmaz,musical moments,given,"study for ""given",chinese opera series,the funeral of romanticism,late afternoon,evening sun,planners,"forwards, parcifal series, group 2, section 4"
0,frank-omeara,Success,"March 30, 1853 ; Carlow, Ireland","October 15, 1888 ; Carlow, Ireland",Irish,Impressionism,painting,Diego Velazquez,Carolus-Duran,"John Singer Sargent , \n John Lavery , \n Carl...",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,goldstein-grigoriy,Success,"December 28, 1878 ; Odessa, Ukraine","May 5, 1938 ; Saint Petersburg, Russian Fede...","Russian , \n Jewish",Symbolism,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,georges-lemmen,Success,"1865 ; Schaerbeek, Belgium","1916 ; Brussels, Belgium",Belgian,"Art Nouveau , \n Neo-Impressionism",painting,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,theodor-aman,Success,"March 20, 1831 ; Câmpulung-Muscel, Romania","August 19, 1891 ; Bucharest, Romania",Romanian,"Academic Art , \n Romanticism","painting , \n engraving , \n drawing",NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,niccolo-cannicci,Success,"1846 ; Firenze, Italy","1906 ; Firenze, Italy",Italian,"Realism , \n Impressionism",painting,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [7]:
import json


wiki_metadata_df2.to_json('wikiart_artist_metadata1.json', orient='records', indent=4)
#result.to_json('wikiart_artist_metadata.json', orient='records', indent=4)


print("Files saved successfully:")
print("- wikiart_artist_metadata1.json")

Files saved successfully:
- wikiart_artist_metadata1.json


In [8]:
import wikipediaapi
import json
import os

wiki_wiki = wikipediaapi.Wikipedia(
    user_agent='ArtInfluenceResearch/1.0 (contact: yourname@example.com)',
    language='en',
    extract_format=wikipediaapi.ExtractFormat.WIKI
)

def fetch_and_save_artist_wiki(page_title, save_dir="./"):
    if not os.path.exists(save_dir):
        os.makedirs(save_dir)
    
    page = wiki_wiki.page(page_title)
    
    if not page.exists():
        print(f"Η σελίδα '{page_title}' δεν βρέθηκε.")
        return None
    
    def get_sections(sections):
        return {s.title: {"text": s.text, "subsections": get_sections(s.sections)} for s in sections}

    artist_data = {
        "title": page.title,
        "summary": page.summary,
        "full_text": page.text,
        "url": page.fullurl,
        "categories": [c.replace("Category:", "") for c in page.categories.keys()],
        "sections": get_sections(page.sections)
    }
    
    file_path = os.path.join(save_dir, f"{page_title.replace(' ', '_')}.json")
    with open(file_path, "w", encoding="utf-8") as f:
        json.dump(artist_data, f, ensure_ascii=False, indent=4)
    
    print(f"Wikipedia Data saved: {file_path}")
    return artist_data

In [11]:
# Εκτέλεση για τον Frank O'Meara
full_data = fetch_and_save_artist_wiki("Frank O'Meara")

if full_data:
    print("\n--- Επισκόπηση Δεδομένων ---")
    print(f"Τίτλος: {full_data['title']}")
    print(f"Σύνολο λέξεων: {len(full_data['full_text'].split())}")
    print(f"Ενότητες που βρέθηκαν: {list(full_data['sections'].keys())}")

Wikipedia Data saved: ./Frank_O'Meara.json

--- Επισκόπηση Δεδομένων ---
Τίτλος: Frank O'Meara
Σύνολο λέξεων: 761
Ενότητες που βρέθηκαν: ['Life', 'Artistic career', 'References', 'External links']


New wikifetch

In [10]:
import json
import time
from tqdm import tqdm

# 1. Load your existing WikiArt metadata
with open('wikiart_artist_metadata1.json', 'r', encoding='utf-8') as f:
    artists_list = json.load(f)

print(f"Loaded {len(artists_list)} artists from JSON.")

# 2. Define the extraction logic
def extract_wiki_title(wiki_field):
    """Extracts 'Frank O'Meara' from 'en.wikipedia.org/wiki/Frank_O'Meara'"""
    if not wiki_field or not isinstance(wiki_field, str):
        return None
    # Split by /wiki/ and take the last part
    if "/wiki/" in wiki_field:
        title = wiki_field.split("/wiki/")[-1]
        # Replace underscores with spaces and decode URL characters
        return title.replace('_', ' ')
    return None

# 3. Loop through everyone
all_wiki_data = []

for artist in tqdm(artists_list, desc="Fetching Wikipedia Data"):
    # Priority 1: Use the Wikipedia URL field if it exists
    page_title = extract_wiki_title(artist.get("wikipedia"))
    
    # Priority 2: Fallback to artist_slug if Wikipedia field is null
    if not page_title:
        # Convert slug 'goldstein-grigoriy' to 'Grigoriy Goldstein'
        slug = artist.get("artist_slug", "")
        page_title = " ".join(word.capitalize() for word in slug.split("-")[::-1])

    # Fetch data
    try:
        wiki_data = fetch_and_save_artist_wiki(page_title, save_dir="./artist_wiki_pages")
        if wiki_data:
            all_wiki_data.append(wiki_data)
        
        # Respectful delay to avoid Wikipedia rate limits
        time.sleep(0.5) 
    except Exception as e:
        print(f"Error processing {page_title}: {e}")

# 4. Final summary
print(f"\nTask Complete. Successfully fetched {len(all_wiki_data)} Wikipedia articles.")

Loaded 1126 artists from JSON.


Fetching Wikipedia Data:   0%|          | 0/1126 [00:00<?, ?it/s]

Wikipedia Data saved: ./artist_wiki_pages/Vasyl_Yermylov.json


Fetching Wikipedia Data:   0%|          | 1/1126 [00:01<34:22,  1.83s/it]

Wikipedia Data saved: ./artist_wiki_pages/Mário_Eloy.json


...

Wikipedia Data saved: ./artist_wiki_pages/Hokkei.json


Fetching Wikipedia Data: 100%|█████████▉| 1125/1126 [22:24<00:00,  1.10it/s]

Η σελίδα 'Shunsen Katsukawa' δεν βρέθηκε.


Fetching Wikipedia Data: 100%|██████████| 1126/1126 [22:25<00:00,  1.19s/it]


Task Complete. Successfully fetched 822 Wikipedia articles.


In [11]:
from SPARQLWrapper import SPARQLWrapper, JSON
import json

# Ρύθμιση του DBpedia Endpoint
sparql = SPARQLWrapper("http://dbpedia.org/sparql")

In [15]:
def fetch_artist_full_knowledge(artist_name):
    resource_slug = artist_name.replace(" ", "_")
    resource_url = f"http://dbpedia.org/resource/{resource_slug}"
    
    sparql = SPARQLWrapper("http://dbpedia.org/sparql")
    
    # Το query τώρα ψάχνει συγκεκριμένα για movement, influencedBy και subjects (categories)
    query = f"""
    PREFIX dbo: <http://dbpedia.org/ontology/>
    PREFIX dbp: <http://dbpedia.org/property/>
    PREFIX dct: <http://purl.org/dc/terms/>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

    SELECT ?rel ?val WHERE {{
      {{ <{resource_url}> dbo:artMovement ?val . BIND("movement" AS ?rel) }}
      UNION
      {{ <{resource_url}> dbp:movement ?val . BIND("movement" AS ?rel) }}
      UNION
      {{ <{resource_url}> dbo:influencedBy ?val . BIND("influencedBy" AS ?rel) }}
      UNION
      {{ <{resource_url}> dbp:knownFor ?val . BIND("knownFor" AS ?rel) }}
      UNION
      {{ <{resource_url}> dct:subject ?val . BIND("category" AS ?rel) }}
      UNION
      {{ <{resource_url}> dbo:wikiPageWikiLink ?val . BIND("connection" AS ?rel) }}
    }}
    """
    
    sparql.setQuery(query)
    sparql.setReturnFormat(JSON)
    
    try:
        results = sparql.query().convert()
        data = {"movement": [], "influencedBy": [], "categories": [], "connections": []}
        
        for res in results["results"]["bindings"]:
            rel = res["rel"]["value"]
            val = res["val"]["value"].split('/')[-1].replace('_', ' ')
            
            if rel == "movement" or rel == "knownFor": data["movement"].append(val)
            elif rel == "influencedBy": data["influencedBy"].append(val)
            elif rel == "category": data["categories"].append(val)
            elif rel == "connection": data["connections"].append(val)
            
        # Καθαρισμός διπλότυπων
        for k in data: data[k] = list(set(data[k]))
        return data
    except Exception as e:
        return None

# Test
frank_knowledge = fetch_artist_full_knowledge("Frank_O'Meara")
print(json.dumps(frank_knowledge, indent=2))

{
  "movement": [
    "Impressionism",
    "Impressionist painting"
  ],
  "influencedBy": [],
  "categories": [
    "Category:Artists from County Carlow",
    "Category:19th-century Irish male artists",
    "Category:19th-century Irish painters",
    "Category:Irish male painters",
    "Category:People educated at St Mary's Knockbeg College",
    "Category:1853 births",
    "Category:People from Carlow (town)",
    "Category:Irish Impressionist painters",
    "Category:1888 deaths"
  ],
  "connections": [
    "Carlow",
    "Category:19th-century Irish male artists",
    "Carolus Duran",
    "En plein air",
    "Ulster Museum",
    "County Carlow",
    "National Gallery of Ireland",
    "Royal Glasgow Institute of the Fine Arts",
    "Kathleen O'Meara (writer)",
    "Crawford Art Gallery",
    "William Stott (artist)",
    "Dublin City Gallery The Hugh Lane",
    "Category:1888 deaths",
    "Impressionism",
    "Grez-sur-Loing",
    "Carl Larsson",
    "Barbizon",
    "Napoleon",
    "

In [16]:
from SPARQLWrapper import SPARQLWrapper, JSON
import json

def fetch_artist_triples(artist_name):
    # Ensure name format is correct for DBpedia URL
    resource_slug = artist_name.replace(" ", "_")
    resource_url = f"http://dbpedia.org/resource/{resource_slug}"
    
    sparql = SPARQLWrapper("http://dbpedia.org/sparql")
    
    # We select EVERYTHING (?p = predicate, ?o = object)
    # We use <{resource_url}> to handle apostrophes safely
    query = f"""
    SELECT ?p ?o WHERE {{
      <{resource_url}> ?p ?o .
    }}
    """
    
    sparql.setQuery(query)
    sparql.setReturnFormat(JSON)
    
    try:
        results = sparql.query().convert()
        triples = []
        
        for res in results["results"]["bindings"]:
            predicate = res["p"]["value"]
            obj = res["o"]["value"]
            
            # We keep the full URIs for the KG, but you can 
            # also store a 'clean' version for display
            triples.append({
                "subject": resource_url,
                "predicate": predicate,
                "object": obj
            })
            
        return triples
    except Exception as e:
        print(f"Error for {artist_name}: {e}")
        return []

# Test for Frank O'Meara
frank_triples = fetch_artist_triples("Frank_O'Meara")

# Print the first 5 triples to see the structure
print(f"Total Triples Found: {len(frank_triples)}")
print(json.dumps(frank_triples[:5], indent=2))

Total Triples Found: 126
[
  {
    "subject": "http://dbpedia.org/resource/Frank_O'Meara",
    "predicate": "http://www.w3.org/1999/02/22-rdf-syntax-ns#type",
    "object": "http://www.w3.org/2002/07/owl#Thing"
  },
  {
    "subject": "http://dbpedia.org/resource/Frank_O'Meara",
    "predicate": "http://www.w3.org/1999/02/22-rdf-syntax-ns#type",
    "object": "http://xmlns.com/foaf/0.1/Person"
  },
  {
    "subject": "http://dbpedia.org/resource/Frank_O'Meara",
    "predicate": "http://www.w3.org/1999/02/22-rdf-syntax-ns#type",
    "object": "http://dbpedia.org/ontology/Person"
  },
  {
    "subject": "http://dbpedia.org/resource/Frank_O'Meara",
    "predicate": "http://www.w3.org/1999/02/22-rdf-syntax-ns#type",
    "object": "http://dbpedia.org/ontology/Person"
  },
  {
    "subject": "http://dbpedia.org/resource/Frank_O'Meara",
    "predicate": "http://www.w3.org/1999/02/22-rdf-syntax-ns#type",
    "object": "http://www.ontologydesignpatterns.org/ont/dul/DUL.owl#NaturalPerson"
  }
]


In [21]:
import pandas as pd
import json
import os

def save_artist_kg(artist_name, triples, folder="./"):
    if not os.path.exists(folder):
        os.makedirs(folder)
    
    # Create a DataFrame for the triples
    kg_df = pd.DataFrame(triples)
    
    # 1. Save the Raw Triples (Full URIs) - Best for Graph DBs
    raw_path = os.path.join(folder, f"{artist_name}_raw_kg.csv")
    kg_df.to_csv(raw_path, index=False)
    
    # 2. Save a "Clean" version for human reading and Bayesian Analysis
    # This removes the long http://dbpedia.org/... prefixes
    clean_df = kg_df.copy()
    clean_df['predicate'] = clean_df['predicate'].apply(lambda x: x.split('/')[-1].split('#')[-1])
    clean_df['object'] = clean_df['object'].apply(lambda x: x.split('/')[-1] if 'http' in str(x) else x)
    
    clean_path = os.path.join(folder, f"{artist_name}_clean_kg.csv")
    clean_df.to_csv(clean_path, index=False)
    
    print(f"Graph nodes saved for {artist_name}:")
    print(f" - Raw: {raw_path}")
    print(f" - Clean: {clean_path}")
    return clean_df

# --- EXECUTION ---
# 1. Fetch
frank_triples = fetch_artist_triples("Frank_O'Meara")

# 2. Save
if frank_triples:
    clean_kg = save_artist_kg("Frank_O'Meara", frank_triples)
    
    # Preview the clean triples
    print("\n--- Knowledge Graph Preview (Cleaned) ---")
    display(clean_kg.head(10))

Graph nodes saved for Frank_O'Meara:
 - Raw: ./Frank_O'Meara_raw_kg.csv
 - Clean: ./Frank_O'Meara_clean_kg.csv

--- Knowledge Graph Preview (Cleaned) ---


,subject,predicate,object
0,http://dbpedia.org/resource/Frank_O'Meara,type,owl#Thing
1,http://dbpedia.org/resource/Frank_O'Meara,type,Person
2,http://dbpedia.org/resource/Frank_O'Meara,type,Person
3,http://dbpedia.org/resource/Frank_O'Meara,type,Person
4,http://dbpedia.org/resource/Frank_O'Meara,type,DUL.owl#NaturalPerson
5,http://dbpedia.org/resource/Frank_O'Meara,type,Q19088
6,http://dbpedia.org/resource/Frank_O'Meara,type,Q215627
7,http://dbpedia.org/resource/Frank_O'Meara,type,Q483501
8,http://dbpedia.org/resource/Frank_O'Meara,type,Q5
9,http://dbpedia.org/resource/Frank_O'Meara,type,Q729


new dbpedia

In [17]:
import pandas as pd
import json
import os
import time
from tqdm import tqdm
from SPARQLWrapper import SPARQLWrapper, JSON
from urllib.parse import unquote

# --- CONFIGURATION ---
sparql = SPARQLWrapper("http://dbpedia.org/sparql")
sparql.setTimeout(30) # Increase timeout for large batches
OUTPUT_FOLDER = "./artist_knowledge_graphs"

# --- HELPER FUNCTIONS ---

def get_dbpedia_resource_name(wiki_field):
    """Extracts the identifier from the Wikipedia URL."""
    if not wiki_field or not isinstance(wiki_field, str):
        return None
    if "/wiki/" in wiki_field:
        # Some URLs in your JSON might not have https://, this split handles both
        resource_part = wiki_field.split("/wiki/")[-1]
        return unquote(resource_part)
    return None

def fetch_artist_triples(resource_name):
    """Queries DBpedia for all triples associated with the artist."""
    # We use <...> to handle names with special characters like apostrophes
    query = f"""
    SELECT ?predicate ?object
    WHERE {{
      <http://dbpedia.org/resource/{resource_name}> ?predicate ?object .
    }}
    """
    sparql.setQuery(query)
    sparql.setReturnFormat(JSON)
    
    try:
        results = sparql.query().convert()
        bindings = results["results"]["bindings"]
        return [{"subject": f"http://dbpedia.org/resource/{resource_name}", 
                 "predicate": b["predicate"]["value"], 
                 "object": b["object"]["value"]} for b in bindings]
    except Exception as e:
        # Return None to indicate a network or query failure
        return None

def save_artist_kg(artist_name, triples, folder):
    """Saves both raw and cleaned versions of the knowledge graph."""
    if not os.path.exists(folder):
        os.makedirs(folder)
    
    # Clean filename for the OS (removing characters that break file paths)
    safe_name = artist_name.replace("'", "").replace('"', "").replace("/", "_").replace(":", "_")
    
    df = pd.DataFrame(triples)
    
    # 1. Save RAW
    raw_path = os.path.join(folder, f"{safe_name}_raw.csv")
    df.to_csv(raw_path, index=False)
    
    # 2. Save CLEAN
    clean_df = df.copy()
    clean_df['predicate'] = clean_df['predicate'].apply(lambda x: x.split('/')[-1].split('#')[-1])
    clean_df['object'] = clean_df['object'].apply(lambda x: x.split('/')[-1] if 'http' in str(x) else x)
    clean_df['subject'] = artist_name
    
    clean_path = os.path.join(folder, f"{safe_name}_clean.csv")
    clean_df.to_csv(clean_path, index=False)

# --- MAIN EXECUTION ---

# 1. Load your full JSON file
with open('wikiart_artist_metadata1.json', 'r', encoding='utf-8') as f:
    artists_list = json.load(f)

print(f"Total artists to process: {len(artists_list)}")

# 2. Loop with error handling and rate limiting
for artist in tqdm(artists_list, desc="Scraping DBpedia"):
    wiki_link = artist.get("wikipedia")
    res_name = get_dbpedia_resource_name(wiki_link)
    
    if res_name:
        # Check if we already have this file (useful if the script crashes and you restart)
        safe_name = res_name.replace("'", "").replace('"', "").replace("/", "_").replace(":", "_")
        if os.path.exists(os.path.join(OUTPUT_FOLDER, f"{safe_name}_raw.csv")):
            continue 

        # Attempt to fetch data
        data = fetch_artist_triples(res_name)
        
        if data:
            save_artist_kg(res_name, data, OUTPUT_FOLDER)
        else:
            # If data is None, there was likely a timeout or a 404
            # We wait a bit longer before next try
            time.sleep(2) 
    
    # Respectful delay between requests
    time.sleep(0.4)

print(f"\nAll done! Data saved in: {OUTPUT_FOLDER}")

Total artists to process: 1126


Scraping DBpedia: 100%|██████████| 1126/1126 [15:58<00:00,  1.18it/s] 


All done! Data saved in: ./artist_knowledge_graphs
